---
---
# 🏗️ Self-Assignment — Mini RAG Project

> **Complete this after class.** Build a small but complete RAG system on a document of your choice, following the same architecture demonstrated in the session.

---

## 🎯 Project Overview

You will build a **Document Q&A Assistant** that:
1. Loads and chunks **your own document** (PDF, TXT, or plain text)
2. Indexes it into ChromaDB using Azure embeddings
3. Answers questions using a **grounded RAG pipeline**
4. Supports **hybrid search** (BM25 + dense)
5. Returns **cited, structured answers**
6. Handles failure modes gracefully

---

## 📐 Architecture You Are Building

```
Your Document (PDF/TXT)
        │
   [Section 1] Load & chunk
        │
   [Section 2] Embed → ChromaDB
        │              │
   User Query ──► [Section 3] Hybrid Search (BM25 + Dense + RRF)
                       │
                  [Section 4] Build grounded prompt
                       │
                  Azure OpenAI (gpt-4o-mini)
                       │
                  [Section 5] Structured answer + citations
                       │
                  [Section 6] Evaluation & reflection
```

---

## 📋 Grading Rubric

| Section | Task | Points |
|---------|------|--------|
| 1 | Document loaded, chunked correctly, chunk stats printed | 15 |
| 2 | Vectors stored in ChromaDB, metadata tagged | 15 |
| 3 | Hybrid search working, BM25 + Dense + RRF fused | 20 |
| 4 | Grounded system prompt, 5 questions answered | 20 |
| 5 | Citation JSON returned and parsed correctly | 15 |
| 6 | At least 2 failure modes demonstrated with fixes | 15 |
| **Total** | | **100** |

---

## 🔖 How to Submit
1. Complete all `# TODO` cells below
2. Run the entire notebook from top to bottom (Runtime → Run All)
3. Save as: `RAG_Assignment_<YourName>.ipynb`
4. Upload to the class portal

> ⚠️ **Important:** All outputs must be visible when you submit. Do NOT clear outputs before submitting.

---
## Step 0 — Choose Your Document

Pick **one** of the following options for your source document.
Choose something you find interesting — you will be asking 5 questions about it!

| Option | Type | Example |
|--------|------|---------|
| A | Company annual/quarterly report | Apple Q4 2024, any public company 10-K |
| B | Research paper or article | Any PDF from arXiv, WHO, World Bank |
| C | Product documentation | API docs, user manual, technical spec |
| D | News article collection | 3–5 articles on the same topic as one text |
| E | Wikipedia article (long-form) | Copy a long Wikipedia article as plain text |

🔑 **Minimum length:** at least 800 words (~5,000 characters) so you get enough chunks to make retrieval meaningful.

In [ ]:
# ── Step 0: Paste or load your document ───────────────────────────────────
#
# Option A — paste text directly:
# MY_DOCUMENT = """
# <paste your document content here>
# """
#
# Option B — load from uploaded file (Google Colab):
# from google.colab import files
# uploaded = files.upload()             # will prompt you to pick a file
# filename = list(uploaded.keys())[0]
# MY_DOCUMENT = uploaded[filename].decode('utf-8')
#
# Option C — load a PDF (requires pypdf):
# !pip install -q pypdf
# from pypdf import PdfReader
# reader = PdfReader("your_file.pdf")
# MY_DOCUMENT = "\n".join(page.extract_text() for page in reader.pages)

# ── TODO: set MY_DOCUMENT to your chosen text ─────────────────────────────
MY_DOCUMENT = """  # <-- REPLACE THIS with your document
"""

# ── TODO: describe your document ──────────────────────────────────────────
DOC_NAME    = "my_document.txt"       # a short filename, e.g. 'apple_q4_2024.pdf'
DOC_TOPIC   = "Describe your document here"  # e.g. 'Apple Q4 2024 earnings report'

# Validation
assert len(MY_DOCUMENT.strip()) >= 500, \
    f"Document too short ({len(MY_DOCUMENT)} chars). Minimum 800 words required."

print(f"✅ Document loaded: '{DOC_TOPIC}'")
print(f"   Name     : {DOC_NAME}")
print(f"   Size     : {len(MY_DOCUMENT):,} characters, ~{len(MY_DOCUMENT.split())} words")

---
## Step 1 — Load & Chunk Your Document

Use the `recursive_split()` function from the demo session.

**Your task:**
- Choose `chunk_size` and `chunk_overlap` appropriate for your document type
- Print chunk statistics (count, average size, min, max)
- Display the first 3 chunks to verify the split looks sensible

**Guidance:**
- Dense prose (reports, articles): `chunk_size=400–512`
- Technical docs with lists: `chunk_size=250–350`
- Always use `chunk_overlap` ≈ 10–15% of `chunk_size`

In [ ]:
# ── Step 1: Chunk your document ───────────────────────────────────────────
# The recursive_split() function is already defined above in the demo section.
# Just call it with appropriate parameters for your document type.

# ── TODO: Set chunk parameters ────────────────────────────────────────────
CHUNK_SIZE    = 0   # TODO: choose a value between 200 and 600
CHUNK_OVERLAP = 0   # TODO: choose overlap ≈ 10–15% of chunk_size

assert CHUNK_SIZE > 0,    "Set CHUNK_SIZE (e.g. 400)"
assert CHUNK_OVERLAP > 0, "Set CHUNK_OVERLAP (e.g. 60)"

# ── TODO: Split the document ──────────────────────────────────────────────
my_chunks = None  # TODO: call recursive_split() here

assert my_chunks is not None and len(my_chunks) >= 3, \
    "my_chunks must contain at least 3 chunks. Check your recursive_split() call."

# ── TODO: Print chunk statistics ─────────────────────────────────────────
# Expected output:
# ✅ Created X chunks
#    Avg size : ??? chars
#    Min size : ??? chars
#    Max size : ??? chars
print("TODO: print chunk statistics")

# ── TODO: Display first 3 chunks ─────────────────────────────────────────
print("\nFirst 3 chunks:")
# TODO: loop over my_chunks[:3] and print each one

---
## Step 2 — Embed & Index into ChromaDB

**Your task:**
- Embed all chunks using the `embed()` helper (Azure API)
- Create a new ChromaDB collection for your document
- Store chunks with meaningful **metadata** tags (at minimum: `source` and `section` or `chunk_index`)
- Run a test search to verify retrieval works

**Tip:** You can reuse `label_section()` from the demo, or write your own labelling logic for your document.

In [ ]:
# ── Step 2a: Embed all chunks via Azure ───────────────────────────────────
import time

print(f"Embedding {len(my_chunks)} chunks...")

# ── TODO: Call embed() and store the result ───────────────────────────────
my_vectors = None  # TODO: call embed(my_chunks)

assert my_vectors is not None, "Call embed(my_chunks) and assign the result to my_vectors"
assert len(my_vectors) == len(my_chunks), "Number of vectors must match number of chunks"

print(f"✅ Embedded {len(my_vectors)} chunks")
print(f"   Vector dimensions : {len(my_vectors[0])}")
print(f"   Preview (chunk 0) : {[round(v, 4) for v in my_vectors[0][:5]]}...")

In [ ]:
# ── Step 2b: Store in ChromaDB ────────────────────────────────────────────
import chromadb, shutil
from pathlib import Path

MY_CHROMA_PATH = "/tmp/my_rag_chroma"
if Path(MY_CHROMA_PATH).exists():
    shutil.rmtree(MY_CHROMA_PATH)

my_chroma_client = chromadb.PersistentClient(path=MY_CHROMA_PATH)

class MyEmbeddingFn(chromadb.EmbeddingFunction):
    def __call__(self, input):
        return embed(input)

my_collection = my_chroma_client.create_collection(
    name="my_document",
    embedding_function=MyEmbeddingFn(),
    metadata={"hnsw:space": "cosine"},
)

# ── TODO: Write a function or logic to assign metadata to each chunk ───────
# At minimum: {"source": DOC_NAME, "chunk_index": i}
# Bonus: add a "section" or "topic" tag using simple keyword logic
def my_metadata(chunk: str, index: int) -> dict:
    # TODO: return a metadata dict for this chunk
    # Example: return {"source": DOC_NAME, "chunk_index": index}
    return {}  # TODO: replace with real metadata

# ── TODO: Add all chunks to the collection ────────────────────────────────
# Hint: use my_collection.add(
#     ids=[...], documents=[...], embeddings=[...], metadatas=[...]
# )
# TODO: add chunks to my_collection

assert my_collection.count() == len(my_chunks), \
    f"Expected {len(my_chunks)} items in collection, got {my_collection.count()}"

print(f"✅ Indexed {my_collection.count()} chunks into ChromaDB")

In [ ]:
# ── Step 2c: Verify search works ─────────────────────────────────────────
# Build your own dense_search function that queries MY collection

def my_dense_search(query: str, k: int = 5, where: dict = None) -> list[dict]:
    """
    Search MY collection. Returns list of {text, score, metadata} dicts.
    """
    # ── TODO: implement this function ─────────────────────────────────────
    # Steps:
    #   1. embed([query]) → query_vector
    #   2. my_collection.query(query_embeddings=[query_vector], ...)
    #   3. return list of {text, score, metadata}
    pass  # TODO: replace with your implementation


# ── TODO: Run a test search using a topic from your document ─────────────
TEST_QUERY = ""  # TODO: fill in a relevant question about your document

assert TEST_QUERY, "Set TEST_QUERY to a question about your document"

test_results = my_dense_search(TEST_QUERY, k=3)

assert test_results and len(test_results) > 0, "my_dense_search() returned no results"

print(f"Test query: '{TEST_QUERY}'")
print()
for i, r in enumerate(test_results):
    print(f"  Rank #{i+1} | score={r['score']:.4f}")
    print(f"  {r['text'][:140]}...")
    print()

---
## Step 3 — Hybrid Search (BM25 + Dense + RRF)

**Your task:**
- Build a BM25 index over `my_chunks`
- Implement `my_bm25_search()` using `rank_bm25`
- Implement `my_hybrid_search()` that fuses BM25 + dense results using the `rrf_fusion()` function from the demo
- Run a comparison: show how BM25, Dense, and Hybrid differ on the same query

**Recall:** `rrf_fusion()` is already defined in the demo section above.

In [ ]:
# ── Step 3a: Build BM25 index ─────────────────────────────────────────────
from rank_bm25 import BM25Okapi
import re

def simple_tokenize(text: str) -> list[str]:
    return re.sub(r'[^\w\s]', ' ', text.lower()).split()

# ── TODO: Build the BM25 index over my_chunks ────────────────────────────
my_bm25_index = None  # TODO: BM25Okapi([simple_tokenize(c) for c in my_chunks])

assert my_bm25_index is not None, "Build the BM25 index"
print("✅ BM25 index built")

In [ ]:
# ── Step 3b: Implement BM25 search ───────────────────────────────────────

def my_bm25_search(query: str, k: int = 5) -> list[dict]:
    """
    BM25 keyword search over my_chunks.
    Returns list of {text, score, metadata, chunk_index} dicts.
    """
    # ── TODO: implement this function ─────────────────────────────────────
    # Steps:
    #   1. tokenize the query with simple_tokenize()
    #   2. get scores: my_bm25_index.get_scores(tokens)
    #   3. sort indices by score descending, take top k
    #   4. return list of dicts with text, score, metadata, chunk_index
    pass  # TODO: replace with your implementation


# ── Step 3c: Implement hybrid search ─────────────────────────────────────

def my_hybrid_search(query: str, k: int = 5, dense_k: int = 10, bm25_k: int = 10) -> list[dict]:
    """
    Hybrid search: BM25 + Dense + RRF fusion.
    Use rrf_fusion() from the demo section above.
    """
    # ── TODO: implement this function ─────────────────────────────────────
    # Steps:
    #   1. dense_hits  = my_dense_search(query, k=dense_k)
    #   2. bm25_hits   = my_bm25_search(query, k=bm25_k)
    #   3. fused       = rrf_fusion([dense_hits, bm25_hits])
    #   4. return fused[:k]
    pass  # TODO: replace with your implementation


# ── TODO: Test with a query where exact keywords matter ──────────────────
KEYWORD_QUERY = ""  # TODO: a query with specific terms/names from your document

assert KEYWORD_QUERY, "Set KEYWORD_QUERY"

print(f"Comparison for: '{KEYWORD_QUERY}'")
print()
d_top = my_dense_search(KEYWORD_QUERY, k=1)
b_top = my_bm25_search(KEYWORD_QUERY, k=1)
h_top = my_hybrid_search(KEYWORD_QUERY, k=1)

assert d_top and b_top and h_top, "All three search functions must return results"

print(f"  Dense  : {d_top[0]['text'][:100]}...")
print(f"  BM25   : {b_top[0]['text'][:100]}...")
print(f"  Hybrid : {h_top[0]['text'][:100]}...")

---
## Step 4 — Build the Grounded RAG Chain

**Your task:**
- Write a system prompt appropriate for your document topic
- Implement a `my_rag()` function that uses `my_hybrid_search()` for retrieval
- Answer **5 meaningful questions** about your document
- Test the refusal: ask one question whose answer is NOT in the document

**Good questions to ask:**
- A specific factual question (a number, date, or name from the document)
- A summary question ("What are the main points of section X?")
- A comparative question ("How does X compare to Y?")
- A causal question ("Why did X happen?")
- An implication question ("What does X mean for the future?")

In [ ]:
# ── Step 4a: Write your system prompt ─────────────────────────────────────

# ── TODO: Write a domain-appropriate system prompt for your document ──────
# Adapt the one from the demo session, but tailor it to your document type.
# Required elements:
#   1. A role description ("You are a ... assistant")
#   2. Grounding rule: "Answer ONLY using the [CONTEXT] below"
#   3. Refusal rule: "If not in context, say 'I don't have enough information'"
#   4. Citation rule: "Cite source after each claim"

MY_SYSTEM_PROMPT = """  # TODO: replace with your system prompt
"""

assert len(MY_SYSTEM_PROMPT.strip()) > 50, \
    "MY_SYSTEM_PROMPT is too short. Write a real grounding prompt."

print("✅ System prompt set")
print(MY_SYSTEM_PROMPT)

In [ ]:
# ── Step 4b: Implement my_rag() ───────────────────────────────────────────

def my_format_context(hits: list[dict]) -> str:
    """Format retrieved chunks into a readable context block."""
    # ── TODO: implement this helper ───────────────────────────────────────
    # For each hit, include: index number, source metadata, and chunk text
    # Separate chunks with a clear divider (e.g. '\n\n---\n\n')
    pass  # TODO: replace


def my_rag(question: str, k: int = 5) -> dict:
    """
    Full RAG pipeline for your document.
    Returns: {question, answer, hits}
    """
    # ── TODO: implement this function ─────────────────────────────────────
    # Steps:
    #   1. hits    = my_hybrid_search(question, k=k)
    #   2. context = my_format_context(hits)
    #   3. messages = [{system prompt}, {user: context + question}]
    #   4. answer  = chat(messages)
    #   5. return {question, answer, hits}
    pass  # TODO: replace


print("✅ my_rag() defined")

In [ ]:
# ── Step 4c: Answer 5 questions about your document ───────────────────────

# ── TODO: Write 5 meaningful questions about your document ────────────────
MY_QUESTIONS = [
    "",   # TODO: Question 1 — specific factual (number, date, name)
    "",   # TODO: Question 2 — summary or main point
    "",   # TODO: Question 3 — comparison or contrast
    "",   # TODO: Question 4 — causal (why did X happen?)
    "",   # TODO: Question 5 — implication or outlook
]

assert all(q.strip() for q in MY_QUESTIONS), \
    "All 5 questions must be filled in (no empty strings)"

# Run all questions
for q in MY_QUESTIONS:
    result = my_rag(q)
    print(f"{'='*68}")
    print(f"Q: {q}")
    print(f"{'─'*68}")
    print(f"A: {result['answer']}")
    print(f"   [Retrieved {len(result['hits'])} chunks | top sim: {result['hits'][0].get('score', result['hits'][0].get('rrf_score', '?')):.4f}]")

In [ ]:
# ── Step 4d: Test out-of-scope refusal ────────────────────────────────────

# ── TODO: Ask something that is clearly NOT in your document ─────────────
OUT_OF_SCOPE_Q = ""  # TODO: e.g. "What happened last year?" or a competitor topic

assert OUT_OF_SCOPE_Q.strip(), "Set OUT_OF_SCOPE_Q"

result = my_rag(OUT_OF_SCOPE_Q)
print(f"Q: {result['question']}")
print(f"A: {result['answer']}")
print()
print("✅ Did the model correctly refuse instead of hallucinating? (check above)")

---
## Step 5 — Structured Citation Output

**Your task:**
- Implement `my_cited_rag()` that returns a **JSON object** with `answer`, `citations`, and `has_sufficient_context` fields
- Run it on 2 of your 5 questions
- Parse and display the citations cleanly

**Expected JSON shape:**
```json
{
  "answer": "Revenue grew 23% ...",
  "citations": [
    {"claim": "Revenue grew 23%", "source": "my_doc.pdf", "section": "executive_summary"}
  ],
  "has_sufficient_context": true
}
```

In [ ]:
# ── Step 5: Cited RAG with structured JSON output ─────────────────────────
import json

# ── TODO: Write a citation system prompt ─────────────────────────────────
# It must:
#   - Instruct the model to return ONLY a JSON object (no markdown, no prose)
#   - Define the exact JSON schema: {answer, citations: [{claim, source, section}], has_sufficient_context}
#   - Instruct the model to only include claims that are in the context
MY_CITATION_SYSTEM = """  # TODO: replace
"""

def my_cited_rag(question: str, k: int = 5) -> dict:
    """
    RAG that returns a structured JSON response with citations.
    Falls back gracefully if JSON parsing fails.
    """
    # ── TODO: implement ───────────────────────────────────────────────────
    # Steps:
    #   1. hits    = my_hybrid_search(question, k=k)
    #   2. context = my_format_context(hits)
    #   3. messages = [{MY_CITATION_SYSTEM}, {user: context + question}]
    #   4. raw    = chat(messages)
    #   5. parse JSON from raw (strip ```json fences if present)
    #   6. return parsed dict (or {answer: raw, citations: [], parse_error: True} on failure)
    pass  # TODO: replace


# ── Run on your first 2 questions ────────────────────────────────────────
for q in MY_QUESTIONS[:2]:
    result = my_cited_rag(q)
    print(f"Q: {q}")
    print(f"A: {result.get('answer', 'N/A')}")
    print(f"Sufficient context: {result.get('has_sufficient_context')}")
    for c in result.get('citations', []):
        print(f"  • '{c.get('claim','')[:60]}' → {c.get('source','')} [{c.get('section','')}]")
    print()

---
## Step 6 — Demonstrate 2 Failure Modes (and Their Fixes)

**Your task:** Choose any **2** of the 4 failure modes below. For each one:
1. Show the **broken** version (incorrect or missing output)
2. Show the **fixed** version (correct output)
3. Write a 1–2 sentence explanation of *why* the fix works

| Failure mode | How to trigger it |
|---|---|
| Hallucination | Use a weak system prompt + ask out-of-scope question |
| Context overflow | Set k to the total number of chunks |
| Format error | Ask for JSON without providing a schema |
| Retrieval failure | Ask with an acronym or synonym that BM25 cannot match |

In [ ]:
# ── Step 6a: Failure Mode 1 ───────────────────────────────────────────────
# ── TODO: Choose one failure mode and demonstrate it ─────────────────────

FAILURE_1_NAME = ""  # TODO: e.g. "Hallucination" or "Context Overflow"

print(f"FAILURE MODE 1: {FAILURE_1_NAME}")
print("="*60)
print()

# TODO: demonstrate the broken behaviour
print("❌ Broken:")
# ... your code here ...

print()
print("✅ Fixed:")
# TODO: demonstrate the fixed behaviour
# ... your code here ...

print()
# ── TODO: Explain why the fix works ──────────────────────────────────────
EXPLANATION_1 = ""  # TODO: 1–2 sentences
print(f"💡 Why the fix works: {EXPLANATION_1}")

In [ ]:
# ── Step 6b: Failure Mode 2 ───────────────────────────────────────────────
# ── TODO: Choose a DIFFERENT failure mode ────────────────────────────────

FAILURE_2_NAME = ""  # TODO: different from FAILURE_1_NAME

assert FAILURE_2_NAME != FAILURE_1_NAME, "Choose a different failure mode for Step 6b"

print(f"FAILURE MODE 2: {FAILURE_2_NAME}")
print("="*60)
print()

print("❌ Broken:")
# TODO: demonstrate the broken behaviour

print()
print("✅ Fixed:")
# TODO: demonstrate the fixed behaviour

print()
EXPLANATION_2 = ""  # TODO: 1–2 sentences
print(f"💡 Why the fix works: {EXPLANATION_2}")

---
## Step 7 — Reflection

Answer the 3 short-answer questions below. Write directly in the markdown cell (double-click to edit).

### ✍️ Reflection Questions

**Q1: What chunk size did you choose and why was it appropriate for your document type?**

> *(your answer here)*

---

**Q2: Did hybrid search return noticeably different results than pure dense search on any of your queries? Describe one example.**

> *(your answer here)*

---

**Q3: If you were deploying this RAG system to real users, what one improvement would you prioritise first and why?**

> *(your answer here)*

In [ ]:
# ── Final submission check ────────────────────────────────────────────────
# Run this cell last. All assertions must pass before submitting.

checks = []

# Step 0
checks.append(("Document loaded",          len(MY_DOCUMENT.strip()) >= 500))
checks.append(("DOC_NAME set",             bool(DOC_NAME.strip())))

# Step 1
checks.append(("Chunks created",           my_chunks is not None and len(my_chunks) >= 3))
checks.append(("Chunk size set",           CHUNK_SIZE > 0 and CHUNK_OVERLAP > 0))

# Step 2
checks.append(("Vectors created",          my_vectors is not None and len(my_vectors) == len(my_chunks)))
checks.append(("ChromaDB indexed",         my_collection.count() == len(my_chunks)))
checks.append(("dense_search working",     bool(my_dense_search(MY_QUESTIONS[0], k=1))))

# Step 3
checks.append(("BM25 index built",         my_bm25_index is not None))
checks.append(("bm25_search working",      bool(my_bm25_search(MY_QUESTIONS[0], k=1))))
checks.append(("hybrid_search working",    bool(my_hybrid_search(MY_QUESTIONS[0], k=1))))

# Step 4
checks.append(("System prompt written",    len(MY_SYSTEM_PROMPT.strip()) > 50))
checks.append(("5 questions filled",       all(q.strip() for q in MY_QUESTIONS)))
checks.append(("my_rag() works",           bool(my_rag(MY_QUESTIONS[0]).get('answer'))))

# Step 5
checks.append(("Citation system prompt",   len(MY_CITATION_SYSTEM.strip()) > 50))

# Step 6
checks.append(("Failure mode 1 named",     bool(FAILURE_1_NAME.strip())))
checks.append(("Failure mode 2 named",     bool(FAILURE_2_NAME.strip())))
checks.append(("Different failure modes",  FAILURE_1_NAME != FAILURE_2_NAME))

# Results
print("\n📋 Submission Checklist")
print("=" * 50)
all_pass = True
for name, result in checks:
    icon = "✅" if result else "❌"
    print(f"  {icon}  {name}")
    if not result:
        all_pass = False

print()
if all_pass:
    print("🎉 All checks passed! Your notebook is ready to submit.")
    print(f"   Save as: RAG_Assignment_{DOC_TOPIC.replace(' ', '_')[:30]}.ipynb")
else:
    print("⚠️  Some checks failed. Fix the ❌ items above before submitting.")